In [ ]:
# clean_eval_champions.ipynb
# Clean the eval_champions battery artifacts so stage 1 re-evaluates.
#
# WHY you'd clean: eval_report.json is cached and report_is_current only
# compares the CHECKPOINT fingerprint -- it cannot detect that the metric CODE
# changed. After editing the battery/metrics, stale reports would be silently
# reused; cleaning them forces a fresh evaluation. Feature caches
# (eval_features_full_fv4.npz, tens of MB per combo) can also be dropped to
# free shared-FS space (they rebuild in ~a minute per combo on GPU).
#
# NEVER touched: checkpoints, manifests, training claims (status.json),
# done/failed markers, probe queues -- this notebook only knows eval files.
import os

APPLY = False                 # dry-run first; flip True and re-run all cells
ONLY_COMBOS = None            # e.g. ['dim018_grl_n2000_view60'] to clean a few;
                              # None = every combo dir under OUT_DIR

CLEAN_REPORTS = True          # <combo>/eval_report.json      -> battery re-runs
CLEAN_FEATURE_CACHES = False  # <combo>/eval_features_full_fv4.npz (re-extract ~1min/combo)
CLEAN_CLAIMS = True           # <combo>/eval_claim.json       -> stray worker claims
CLEAN_AGGREGATES = True       # grid_metrics.json + champions.json (+ stray .tmp.*)
CLEAN_SHARED_CACHES = False   # raw_identity/description/text-variant embedding caches
                              # (expensive re-embeds). NOTE: tag_text_eval_split.json is
                              # NEVER deleted -- a new random split would make variant
                              # metrics incomparable across combos evaluated before/after.

REPO = "/workspace/stable-query-latent"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
ROOT = os.path.join(REPO, OUT_DIR)

print('apply :', APPLY)
print('root  :', ROOT)
print('scopes:', {k: v for k, v in [('reports', CLEAN_REPORTS), ('feature_caches', CLEAN_FEATURE_CACHES),
                                    ('claims', CLEAN_CLAIMS), ('aggregates', CLEAN_AGGREGATES),
                                    ('shared_caches', CLEAN_SHARED_CACHES)]})


In [ ]:
# Scan: local battery workers + deletable files by category (counts + sizes).
import glob, json, time
from pathlib import Path
import psutil

WORKER_MARK = 'eval_battery_worker.py'


def local_battery_procs():
    procs = []
    for p in psutil.process_iter(['pid', 'cmdline']):
        try:
            cmd = ' '.join(p.info['cmdline'] or [])
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
        if WORKER_MARK in cmd:
            procs.append((p.info['pid'], cmd[:150]))
    return procs


root = Path(ROOT)
per_combo_names = []
if CLEAN_REPORTS:
    per_combo_names.append('eval_report.json')
if CLEAN_FEATURE_CACHES:
    per_combo_names.append('eval_features_full_fv4.npz')
if CLEAN_CLAIMS:
    per_combo_names.append('eval_claim.json')

targets = {'reports': [], 'feature_caches': [], 'claims': [], 'aggregates': [], 'shared_caches': []}
_name_to_cat = {'eval_report.json': 'reports',
                'eval_features_full_fv4.npz': 'feature_caches',
                'eval_claim.json': 'claims'}

if root.exists():
    for d in sorted(root.iterdir()):
        if not d.is_dir() or d.name == 'VM_parallel':
            continue
        if ONLY_COMBOS is not None and d.name not in ONLY_COMBOS:
            continue
        for name in per_combo_names:
            f = d / name
            if f.exists():
                targets[_name_to_cat[name]].append(f)
    if CLEAN_AGGREGATES and ONLY_COMBOS is None:
        for pat in ('grid_metrics.json', 'champions.json',
                    'grid_metrics.json.tmp.*', 'champions.json.tmp.*'):
            targets['aggregates'] += [Path(p) for p in glob.glob(str(root / pat))]
    if CLEAN_SHARED_CACHES and ONLY_COMBOS is None:
        for pat in ('raw_identity_cache_ms*.npz', 'description_raw_qwen_ms*.npz',
                    'text_variant_embedding_cache.npz'):
            targets['shared_caches'] += [Path(p) for p in glob.glob(str(root / pat))]

procs = local_battery_procs()
print(f'{len(procs)} local battery worker(s):')
for pid, cmd in procs:
    print(f'  kill pid {pid}: {cmd}')
total = 0
for cat, files in targets.items():
    size = sum(f.stat().st_size for f in files if f.exists())
    total += size
    print(f'{cat:15}: {len(files):4d} file(s)  {size / 2**20:10.1f} MiB')
print(f'{"TOTAL":15}: {sum(len(v) for v in targets.values()):4d} file(s)  {total / 2**20:10.1f} MiB')
if not APPLY:
    print()
    print('DRY-RUN: nothing killed or deleted. Set APPLY=True and re-run all cells.')


In [ ]:
# Apply: stop local battery workers, delete per scope, report exact counts.
if not APPLY:
    print('APPLY=False -- skipped.')
else:
    import signal

    for pid, _cmd in procs:
        try:
            os.kill(pid, signal.SIGTERM)
        except OSError:
            pass
    time.sleep(3)
    killed = 0
    for pid, _cmd in procs:
        try:
            os.kill(pid, signal.SIGKILL)
            killed += 1
            print(f'SIGKILLed worker {pid}')
        except (OSError, ProcessLookupError):
            killed += 1                      # exited on SIGTERM

    deleted = {}
    for cat, files in targets.items():
        n = 0
        for f in files:
            try:
                Path(f).unlink()
                n += 1
            except OSError:
                pass
        deleted[cat] = n

    print()
    print(f'summary: stopped {killed}/{len(procs)} worker(s); deleted ' +
          '  '.join(f'{cat}={n}' for cat, n in deleted.items()))
    if deleted.get('reports'):
        print('re-run eval_champions.ipynb to re-evaluate; grid_metrics.json will be '
              'republished streamingly as fresh reports land.')
